In [1]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV (GUJARAT)
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/GJ_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)

features = ['NDVI', 'NDMI', 'NBR', 'BSI']
df = df[['latitude', 'longitude', 'year'] + features]
df = df.dropna()
df = df.sort_values(by=['latitude', 'longitude', 'year'])

# ===============================
# 3. Create Sequences (RAW DATA)
# ===============================
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

for (lat, lon), group in df.groupby(['latitude', 'longitude']):
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/GJ_Deforestation_Predictions.csv',
    index=False
)

print("Saved: GJ_Deforestation_Predictions.csv")
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/GJ_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/GJ_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Mounted at /content/drive
Raw input shape: (17338, 4, 4)

Label distribution:
No deforestation: 15699
Deforestation: 1639
Train shape: (13870, 4, 4)
Test shape : (3468, 4, 4)
Scaled train shape: (13870, 4, 4)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.1786 - loss: 1.3753 - precision: 0.1001 - recall: 0.9626 - val_accuracy: 0.5868 - val_loss: 0.6856 - val_precision: 0.1748 - val_recall: 0.9055
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7384 - loss: 0.9427 - precision: 0.2431 - recall: 0.8368 - val_accuracy: 0.8893 - val_loss: 0.2705 - val_precision: 0.4529 - val_recall: 0.8201
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8371 - loss: 0.6317 - precision: 0.3571 - recall: 0.9047 - val_accuracy: 0.8080 - val_loss: 0.4219 - val_precision: 0.3268 - val_recall: 0.9726
Epoch 4/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8650 - loss: 0.5348 - precision: 0.4060 - recall: 0.9245 - val_accuracy: 0.9112 - val_loss: 0.2346 - val_precision: 0.5171 - val_recall: 0.9238
Epoch 5/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8844 - loss: 0.4706 - precision: 0.4468 - recall: 0.9390 - val_accuracy: 0.8567 - val_loss

In [2]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/GJ_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.1136949241252307), 'std': 0.11071283334855499, 'q1': np.float64(-0.18331186749999998), 'q3': np.float64(-0.0459645675), 'lower_2std': np.float64(-0.3351205908223407), 'upper_2std': np.float64(0.10773074257187927)}
NBR : {'mean': np.float64(-0.08861397298995478), 'std': 0.12706162301919127, 'q1': np.float64(-0.15902119999999997), 'q3': np.float64(-0.01707311999999997), 'lower_2std': np.float64(-0.3427372190283373), 'upper_2std': np.float64(0.16550927304842777)}
BSI : {'mean': np.float64(0.03744985462468935), 'std': 0.08943410751679458, 'q1': np.float64(-0.015643004694321224), 'q3': np.float64(0.08780589421769341), 'lower_2std': np.float64(-0.14141836040889982), 'upper_2std': np.float64(0.2163180696582785)}
NDMI: {'mean': np.float64(-0.041768143526286015), 'std': 0.09936171857413095, 'q1': np.float64(-0.09557050249999999), 'q3': np.float64(0.016038447499999997), 'lower_2std': np.float64(-0.24049158067454793), 'upper_2std': np.float64(0.15695529362197588)}

De

In [3]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# =====================================
# Gujarat Map
# Gujarat roughly:
# Latitude 20.0 – 24.7
# Longitude 68.0 – 74.5
# =====================================

m = folium.Map(location=[22.5, 71.5], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/GJ_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())

# Cause color mapping
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}

# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)



cause
Logging        338
Fire           117
Agriculture     52
Unknown         39
Urban/Other      1
Name: count, dtype: int64


In [4]:
m.save('/content/drive/MyDrive/GJ_adaptive.html')


In [5]:
m

In [6]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS (GUJARAT)
# ==============================

df_def = pd.read_csv(
    '/content/drive/MyDrive/GJ_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))


# ==============================
# LOAD CAUSE DATA (GUJARAT)
# ==============================

df_cause = pd.read_csv(
    '/content/drive/MyDrive/GJ_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


# ==============================
# LOAD GUJARAT DISTRICTS
# ==============================

gj = gpd.read_file('/content/drive/MyDrive/Gujarat.geojson')

# CRS SAFETY
if gj.crs is None:
    gj.set_crs("EPSG:4326", inplace=True)

gj = gj.to_crs("EPSG:4326")

# Optional label
gj["state"] = "Gujarat"

print(gj.head())


# ==============================
# MERGE CAUSE DATA
# ==============================

df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


# ==============================
# CREATE POINT GEOMETRY
# ==============================

geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 3468
    latitude  longitude  deforestation
0  21.477416  70.527137              0
1  21.537513  74.284790              0
2  23.167147  73.706275              0
3  21.234871  72.893300              1
4  22.657802  73.356920              0
Total deforestation samples: 547
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  21.234871  72.893300              1    -0.057264   -0.415349    0.163067   
1  21.886958  73.491578              1    -0.232141   -0.268635    0.206795   
2  21.642706  72.831406              1    -0.407347   -0.451731    0.287149   
3  21.236578  72.793587              1    -0.036325   -0.437651    0.180338   
4  21.309341  73.003792              1    -0.197919   -0.201794    0.142819   

   NDMI_change    cause  
0    -0.266926     Fire  
1    -0.223330  Logging  
2    -0.311011     Fire  
3    -0.302697     Fire  
4    -0.137626  Logging  
  REMARKS_2 Country State_Name State_Code     Dist_Name Dist_Code  \
0      None  

In [7]:
# ==============================
# SPATIAL JOIN (Points → Gujarat districts)
# ==============================

gdf_joined = gpd.sjoin(
    gdf_points,
    gj,                 # ← Gujarat district polygons
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())


# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================

if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# cleanup
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)


# ==============================
# DISTRICT SUMMARY — Gujarat
# ==============================

district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/GJ_District_Wise_Deforestation.csv',
    index=False
)

print("Saved GJ district summary")


# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================

cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/GJ_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved GJ cause summary")

    latitude  longitude  deforestation    cause                   geometry  \
0  21.234871  72.893300              1     Fire   POINT (72.8933 21.23487)   
1  21.886958  73.491578              1  Logging  POINT (73.49158 21.88696)   
2  21.642706  72.831406              1     Fire  POINT (72.83141 21.64271)   
3  21.236578  72.793587              1     Fire  POINT (72.79359 21.23658)   
4  21.309341  73.003792              1  Logging  POINT (73.00379 21.30934)   

   index_right REMARKS_2 Country State_Name State_Code Dist_Name Dist_Code  \
0         32.0      None   India    Gujarat         24     Surat       492   
1          9.0      None   India    Gujarat         24   Narmada       487   
2          2.0      None   India    Gujarat         24   Bharuch       488   
3         32.0      None   India    Gujarat         24     Surat       492   
4         32.0      None   India    Gujarat         24     Surat       492   

     state  
0  Gujarat  
1  Gujarat  
2  Gujarat  
3  Gujarat

In [9]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Gujarat District Boundaries
# =====================================================
gj = gpd.read_file('/content/drive/MyDrive/Gujarat.geojson')

# CRS safety
if gj.crs is None:
    gj.set_crs(epsg=4326, inplace=True)

gj = gj.to_crs(epsg=4326)
gj["state"] = "Gujarat"
gdf_districts = gj.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/GJ_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0  # placeholder if no data

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join (assign districts to points)
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats into District Map
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_deforestation, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_afforestation, on="Dist_Name", how="left"
)

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

# Deforestation
gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)
gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)
gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

# Afforestation
gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)
gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)
gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Folium Map (center Gujarat)
# =====================================================
m = folium.Map(location=[22.3, 71.2], zoom_start=6)

# =====================================================
# STEP 10: Dynamic Choropleth for Deforestation
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Gujarat)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Dynamic Gujarat Afforestation Alert Popup
# =====================================================
state_total = gdf_districts["total_samples"].sum()
state_def = gdf_districts["deforestation_samples"].sum()
state_aff = gdf_districts["afforestation_samples"].sum()

# Get Top 3 districts by deforestation
top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

# Build dynamic district list
top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Gujarat Afforestation Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Immediate afforestation drives recommended.<br>
Focus on plantation, controlled logging,<br>
and sustainable land management.
"""

folium.Marker(
    location=[22.3, 71.2],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/gj_forest.html")

print("✅ Gujarat map saved successfully with afforestation recommendation!")

/tmp/ipykernel_4761/3224139603.py:88: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Gujarat map saved successfully with afforestation recommendation!
